In [9]:
import pandas as pd
from glob import glob
import biotite.structure.io as bsio
import numpy as np
from superimpose import superimpose_chain
from pdb_utils import get_biotite_ligand_as_rdmol
from rdkit.Chem.rdMolAlign import CalcRMS
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

Read the data generated by `pxr_preprocess.ipynb`

In [17]:
df = pd.read_parquet("pxr_clean_ligand_info_and_sequences.parquet")

In [18]:
df.head()

,pdb,chain,cmp_id,SMILES,num_cif_atoms,num_smi_atoms,sequence
0,1ilh,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33,MKKGHHHHHHGSERTGTQPLGVQGLTEEQRMMIRELMDAQMKTFDT...
1,1m13,A,HYF,O=C1C2(C(O)=C(C(=O)C1(CC(C\C=C(/C)C)C2(C)CC\C=...,39,39,MKKGHHHHHHGSERTGTQPLGVQGLTEEQRMMIRELMDAQMKTFDT...
2,1nrl,A,SRL,O=P(OCC)(OCC)/C(=C\c1cc(c(O)c(c1)C(C)(C)C)C(C)...,33,33,MKKGHHHHHHGSERTGTQPLGVQGLTEEQRMMIRELMDAQMKTFDT...
3,2o9i,A,444,O=S(=O)(N(c1ccc(cc1)C(O)(C(F)(F)F)C(F)(F)F)CC(...,31,31,GLTEEQRMMIRELMDAQMKTFDTTFSHFKNFRLPGVLSSGCELPES...
4,2qnv,A,CDZ,O=C(C1=C(O)C(=C(O)C(C1=O)(C\C=C(/C)C)C\C=C(/C)...,29,29,MKKGHHHHHHGSERTGTQPLGVQGLTEEQRMMIRELMDAQMKTFDT...


Loop over structures
- Read the Boltz-2 structure
- Read the reference cif file
- Superimpose the protein structures
- Extract the Boltz-2 ligand structure as an RDKit molecule
- Extract the reference ligand structure as an RDKit molecule
- Calculate the RMSD bewteen the two RDKit molecules

In [19]:
base_dir = "/Users/pwalters/DATA/BOLTZ/PXR_2025_12_03/boltz_results_{pdb}/predictions/{pdb}"
result_list = []
for pdb, chain, cmp_id, smiles in tqdm(df[["pdb","chain","cmp_id","SMILES"]].values):
    boltz_dir = f"/Users/pwalters/DATA/BOLTZ/PXR_2025_12_03/boltz_results_{pdb}/predictions/{pdb}"
    for filename in sorted(glob(f"{boltz_dir}/*.cif")):
        boltz_atoms = bsio.load_structure(filename)
        # fix the boltz chain ids
        boltz_atoms.chain_id = np.char.strip(boltz_atoms.chain_id, '[]')
        # read the reference file
        ref_atoms = bsio.load_structure(f"{pdb}.cif")
        # limit the reference file to the specified chain
        mask = (ref_atoms.chain_id == chain)
        ref_atoms = ref_atoms[mask]
        # superimpose the protein structures
        boltz_atoms,_,_,_ = superimpose_chain(ref_atoms, boltz_atoms)
        # get the ligand structures as RDKit molecules
        ref_rd_mol = get_biotite_ligand_as_rdmol(ref_atoms,chain,cmp_id,smiles)
        boltz_rd_mol = get_biotite_ligand_as_rdmol(boltz_atoms,"B","LIG1",smiles)
        # report the RMSD
        result_list.append([pdb,CalcRMS(ref_rd_mol,boltz_rd_mol)])

  0%|          | 0/56 [00:00<?, ?it/s]

/opt/homebrew/Caskroom/miniforge/base/envs/rdkit_2025_10/lib/python3.11/site-packages/biotite/structure/io/pdbx/convert.py:461: UserWarning: Attribute 'auth_atom_id' not found within 'atom_site' category. The fallback attribute 'label_atom_id' will be used instead
  warnings.warn(


In [16]:
result_df = pd.DataFrame(result_list,columns=["PDB","RMSD"])
result_df.sort_values("RMSD")

,PDB,RMSD
32,7riu,0.636613
3,2o9i,0.752676
18,6nx1,0.939459
41,8r00,1.159730
53,9fzh,1.488712
31,7rio,1.493006
20,6s41,1.687153
28,7axe,1.777427
16,6hj2,1.824857
19,6p2b,1.879321
